# 3. Feature Engineering

This notebook merges fire and weather datasets, then engineers temporal features:
- 7-day rolling mean: temperature, wind speed, VPD
- 14-day cumulative precipitation

> **Prerequisite:** Run Notebooks 01 and 02 first.

## Merge Fire + Weather Data

In [ ]:
dataset = dataset.rename(columns={"acq_date": "date"})
dataset["date"] = pd.to_datetime(dataset["date"])

In [ ]:
full_dataset = dataset.merge(
    weather_df,
    on=["lat_bin", "lon_bin", "date"],
    how="inner"
)

print("Merged rows:", len(full_dataset))
full_dataset.head()

In [ ]:
full_dataset["prev_7_fire"] = full_dataset["prev_7_fire"].fillna(0)

## Select Model Features

In [ ]:
model_df = full_dataset[
    [
        "lat_bin", "lon_bin", "date",
        "temp_c", "precip_mm", "wind_speed",
        "relative_humidity", "vpd",
        "ignition"
    ]
].copy()

print(model_df.head())
print("Final rows:", len(model_df))

## Rolling Temporal Features

In [ ]:
# Sort properly
model_df = model_df.sort_values(["lat_bin", "lon_bin", "date"])

# Group by spatial cell
grouped = model_df.groupby(["lat_bin", "lon_bin"])

# 7-day rolling temperature mean
model_df["temp_7d_mean"] = grouped["temp_c"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)

# 7-day rolling wind mean
model_df["wind_7d_mean"] = grouped["wind_speed"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)

# 7-day rolling VPD mean
model_df["vpd_7d_mean"] = grouped["vpd"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)

# 14-day cumulative rainfall
model_df["precip_14d_sum"] = grouped["precip_mm"].transform(
    lambda x: x.shift(1).rolling(14, min_periods=1).sum()
)

In [ ]:
model_df[[
    "temp_7d_mean",
    "wind_7d_mean",
    "vpd_7d_mean",
    "precip_14d_sum"
]].head(10)

## Clean & Finalize

In [ ]:
model_df = model_df.dropna().copy()

print("Rows after dropping NaNs:", len(model_df))

In [ ]:
final_df = model_df[
    [
        "temp_7d_mean",
        "wind_7d_mean",
        "vpd_7d_mean",
        "precip_14d_sum",
        "ignition"
    ]
].copy()

print(final_df.head())
print("Final rows:", len(final_df))
print("Ignition positives:", final_df["ignition"].sum())